In [ ]:
# Step 1: Imports
import os
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset

os.environ["WANDB_DISABLED"] = "true"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(" Device:", device)


file_path = r"C:\\Users\\nardo\\Downloads\\AzureLLMInferenceTrace_conv (1).csv"
df = pd.read_csv(file_path, engine="python", on_bad_lines="skip")
df.columns = [col.strip() for col in df.columns]
df = df.dropna(subset=["ContextTokens", "GeneratedTokens"])
# Step 4: Input formatting
df['TrainText'] = df.apply(lambda row: f"Context: {row['ContextTokens']} tokens, Generated: {row['GeneratedTokens']} tokens", axis=1)
df['TestText'] = df['ContextTokens'].apply(lambda x: f"Context: {x} tokens")

train_df, test_val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(test_val_df, test_size=0.5, stratify=test_val_df['label'], random_state=42)
train_texts = train_df['TrainText'].tolist()
val_texts = val_df['TrainText'].tolist()
test_texts = test_df['TestText'].tolist()  # Only context at test time
# this part it is passing argumnt to the class and object constractor and this are objects 
train_dataset = TextDataset(train_texts, train_df['label'].tolist())
val_dataset = TextDataset(val_texts, val_df['label'].tolist())
test_dataset = TextDataset(test_texts, test_df['label'].tolist())
#  Step 7: Model & Trainer
model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased").to(device)
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=6,
    fp16=torch.cuda.is_available(),
    report_to="none"
)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)
#  Step 8: Train
trainer.train()

# Step 9: Evaluate
preds = trainer.predict(test_dataset)
y_true = test_df['label'].tolist()
y_pred = preds.predictions.argmax(axis=1)
print(" Classification Report:")
